In [ ]:
!pip install pydantic pydantic[email]

In [1]:
import json
from typing import Optional, Literal
from pydantic import BaseModel, Field, EmailStr, field_validator, model_validator, computed_field, ValidationError

In [2]:
class Customer(BaseModel):
    name: str
    email: EmailStr
    phone: Optional[str] = None

    @field_validator("name", mode="before")
    @classmethod
    def to_title(cls, v):
        return v.title()

In [3]:
class Product(BaseModel):
    product_id: str
    name: str
    price: float = Field(gt=0)

    @field_validator("name", mode="before")
    @classmethod
    def to_title(cls, v):
        return v.title()

In [4]:
class OrderItem(BaseModel):
    product: Product
    quantity: int = Field(ge=1)

    @computed_field
    def line_total(self) -> float:
        return round(self.product.price * self.quantity, 2)

In [5]:
class Order(BaseModel):
    order_id: str
    customer: Customer
    items: list[OrderItem]
    status: Literal["pending", "confirmed", "shipped", "delivered", "cancelled"]
    total: float

    @model_validator(mode="after")
    def check_total(self):
        calculated = round(sum(item.line_total for item in self.items), 2)
        if round(self.total, 2) != calculated:
            raise ValueError(f"Total mismatch: provided {self.total}, calculated {calculated}")
        return self

    @computed_field
    def total_with_tax(self) -> float:
        return round(self.total * 1.10, 2)

In [6]:
with open("orders_raw.json", "r") as f:
    raw_orders = json.load(f)

print(f"Loaded {len(raw_orders)} orders")

Loaded 6 orders


In [ ]:
validated_orders = []
errors = []

for raw in raw_orders:
    order_id = raw.get("order_id", "Unknown")
    try:
        order = Order.model_validate(raw)
        validated_orders.append(order)
        print(f"  Pass  {order_id}")
    except ValidationError as e:
        error_list = []
        for err in e.errors():
            loc = " -> ".join(str(x) for x in err["loc"])
            error_list.append({"field": loc, "reason": err["msg"]})
        errors.append({"order_id": order_id, "errors": error_list})
        print(f"  Fail  {order_id} — {len(error_list)} error")

print(f"\n{len(validated_orders)} valid, {len(errors)} failed")

  PASS  ORD-001
  PASS  ORD-002
  PASS  ORD-003
  FAIL  ORD-004 — 1 error(s)
  FAIL  ORD-005 — 1 error(s)
  FAIL  ORD-006 — 1 error(s)

3 valid, 3 failed


In [ ]:
o = validated_orders[0]
print("Order ID:", o.order_id)
print("Customer:", o.customer.name, " ", o.customer.email)
print("Status:", o.status)
print("Total:", o.total)
print("Total w/Tax:", o.total_with_tax)
print()
for item in o.items:
    print(f"  {item.product.name} x{item.quantity} @ ${item.product.price} = ${item.line_total}")

Order ID     : ORD-001
Customer     : John Doe | john.doe@example.com
Status       : pending
Total        : 134.98
Total w/ Tax : 148.48

  Wireless Headphones x2 @ $59.99 = $119.98
  Phone Case x1 @ $15.0 = $15.0


In [9]:
for entry in errors:
    print(f"Order: {entry['order_id']}")
    for err in entry["errors"]:
        print(f"  Field : {err['field']}")
        print(f"  Reason: {err['reason']}")
    print()

Order: ORD-004
  Field : customer -> email
  Reason: value is not a valid email address: An email address must have an @-sign.

Order: ORD-005
  Field : items -> 0 -> product -> price
  Reason: Input should be greater than 0

Order: ORD-006
  Field : 
  Reason: Value error, Total mismatch: provided 999.0, calculated 22.0



In [10]:
validated_list = [json.loads(order.model_dump_json()) for order in validated_orders]

with open("validated_orders.json", "w") as f:
    json.dump(validated_list, f, indent=2)

with open("error_report.json", "w") as f:
    json.dump(errors, f, indent=2)

print(f"validated_orders.json — {len(validated_list)} orders")
print(f"error_report.json     — {len(errors)} errors")

validated_orders.json — 3 orders
error_report.json     — 3 errors
